# Level 2 — Feature Engineering with DuckDB

## Objective

This notebook loads the cleaned dataset into DuckDB and creates engineered features that support downstream business analysis. These features enrich the original dataset by deriving new metrics, validating their correctness, and preparing the data for more advanced analytics in the next notebook.

## Load Dataset into DuckDB     

Load the cleaned dataset into DuckDB to begin feature engineering and analytical processing. 

In [1]:
from pathlib import Path
import duckdb

# Path to the cleaned dataset
data_path = Path("../data/superstore_utf8.csv")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE VIEW superstore AS
SELECT *
FROM read_csv_auto('{data_path.as_posix()}', header=True)
""")

con.execute("""
SELECT *
FROM superstore
LIMIT 5
""").df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Feature Engineering

### Feature Creation
A DuckDB view named `superstore_features` was created to preserve the original dataset while adding analytical features. New columns include `fulfillment_days`, `profit_margin`, `order_year`, `order_month`, and `customer_lifetime_sales` (calculated using a window function). These engineered features support customer segmentation, profitability analysis, and operational performance reporting. # I want to change the term customer segmentation here

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    date_diff('day', order_date, ship_date) AS fulfillment_days,
    profit / sales AS profit_margin,
    year(order_date) AS order_year,
    month(order_date) AS order_month,
    SUM(sales) OVER (
        PARTITION BY customer_id
    ) AS customer_lifetime_sales
FROM superstore
""") # should we keep this output view? 

### Quick Inspection of Engineered Features

The newly created features were queried from the `superstore_features` view and inspected using a sample of 10 records. This validation step ensured that the feature engineering process produced the expected values before proceeding with further analysis.

In [15]:
con.execute("""
SELECT
    customer_name,
    order_id,
    fulfillment_days,
    profit_margin,
    order_year,
    order_month,
    customer_lifetime_sales
FROM superstore_features
ORDER BY order_date
LIMIT 10;
""").df()

,customer_name,order_id,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales
0,Darren Powers,CA-2014-103800,4,0.3375,2014,1,1050.636
1,Phillina Ober,CA-2014-112326,4,-0.2375,2014,1,1056.858
2,Phillina Ober,CA-2014-112326,4,0.3625,2014,1,1056.858
3,Phillina Ober,CA-2014-112326,4,-1.5500,2014,1,1056.858
4,Mick Brown,CA-2014-141817,7,0.2500,2014,1,1428.231
5,Maria Etezadi,CA-2014-167199,4,0.4500,2014,1,10663.728
6,Maria Etezadi,CA-2014-167199,4,0.4600,2014,1,10663.728
7,Maria Etezadi,CA-2014-167199,4,0.2700,2014,1,10663.728
8,Maria Etezadi,CA-2014-167199,4,0.0100,2014,1,10663.728
9,Maria Etezadi,CA-2014-167199,4,0.2900,2014,1,10663.728


## Feature Validation

### Fulfillment Days Validation

The distribution of the engineered `fulfillment_days` feature was examined to verify that the calculated values were reasonable. Most orders were fulfilled within **4–5 days**, with relatively few orders requiring **0–1 days** or the maximum of **7 days**, indicating a realistic distribution suitable for downstream operational analysis.

In [4]:
con.execute('''
SELECT
    fulfillment_days,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY fulfillment_days
ORDER BY fulfillment_days;
''').df()            

,fulfillment_days,orders
0,0,519
1,1,369
2,2,1334
3,3,1005
4,4,2774
5,5,2169
6,6,1203
7,7,621


### Profit Margin Validation

Analysis of the engineered `profit_margin` feature showed an average profit margin of **12.03%**, indicating that the company earned approximately 12 cents of profit for every dollar of sales. Profit margins ranged from **-275%** to **50%**, highlighting that while some transactions were highly profitable, others resulted in substantial losses.

In [5]:
con.execute('''
SELECT
    ROUND(AVG(profit_margin),4) AS avg_profit_margin,
    MIN(profit_margin) AS min_margin,
    MEDIAN(profit_margin) AS median_margin,
    MAX(profit_margin) AS max_margin
FROM superstore_features;
''').df()

,avg_profit_margin,min_margin,median_margin,max_margin
0,0.1203,-2.75,0.27,0.5


**Summary:** 

The products with the lowest profit margins were identified to better understand the transactions contributing to overall losses. Several products exhibited profit margins below **-270%**, indicating that the losses incurred on these sales substantially exceeded the revenue generated, making them potential candidates for further pricing or discount analysis.


#I think this is better than ### Lowest Profit Margin Products

### Order Year Validation 

The engineered `order_year` feature was validated by counting the number of orders in each year. The results confirmed that the year was extracted correctly from the original order dates.

In [14]:
con.execute('''
SELECT
    order_year,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
''').df()

,order_year,orders
0,2014,1993
1,2015,2102
2,2016,2587
3,2017,3312


### Order Month Validation 

The engineered `order_month` feature was validated by summarizing the number of orders placed in each month. This confirmed that the month values were extracted correctly and are available for seasonal analysis.

In [13]:
con.execute('''
SELECT
    order_month,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_month
ORDER BY order_month;
''').df()

,order_month,orders
0,1,381
1,2,300
2,3,696
3,4,668
4,5,735
5,6,717
6,7,710
7,8,706
8,9,1383
9,10,819


### Customer Lifetime Sales Validation

The `customer_lifetime_sales` feature was used to identify the highest-value customers in the dataset. Sean Miller generated over **$25,000** in lifetime sales, while several other customers exceeded **$12,000** in total purchases, indicating that a relatively small group of customers contributed substantially to overall revenue.

In [7]:
con.execute('''
SELECT DISTINCT
    customer_name,
    customer_lifetime_sales
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,customer_lifetime_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


## Customer Value Classification

Customer lifetime sales were further transformed into a categorical feature to simplify the identification of high-value customers. This business-oriented classification enables more intuitive customer comparisons in downstream analyses.

### Determine Classification Thresholds

Percentiles of the `customer_lifetime_sales` distribution were calculated to establish data-driven thresholds for customer value classification. Using these statistical breakpoints ensures that customer tiers are based on the underlying distribution of the data rather than arbitrary spending thresholds.

In [8]:
con.execute('''
SELECT
    quantile_cont(customer_lifetime_sales, 0.25) AS q1,
    quantile_cont(customer_lifetime_sales, 0.50) AS median,
    quantile_cont(customer_lifetime_sales, 0.75) AS q3,
    quantile_cont(customer_lifetime_sales, 0.9) AS top_10,
FROM (
    SELECT DISTINCT
        customer_id,
        customer_lifetime_sales
    FROM superstore_features
);
''').df()

,q1,median,q3,top_10
0,1146.05,2256.394,3785.276,6038.48


### Create the `customer_tier` Feature

A new feature, `customer_tier`, was engineered by applying the previously calculated percentile thresholds to each customer's lifetime sales. Using a `CASE` statement, customers were classified into Standard, High Value, Premium, and Big Fish tiers, creating a reusable business feature for downstream customer analysis.

In [9]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    CASE
        WHEN customer_lifetime_sales >= 6038.48 THEN 'Big Fish'
        WHEN customer_lifetime_sales >= 3785.276 THEN 'Premium'
        WHEN customer_lifetime_sales >= 2256.394 THEN 'High Value'
        ELSE 'Standard'
    END AS customer_tier
FROM (
    SELECT
        *,
        date_diff('day', order_date, ship_date) AS fulfillment_days,
        profit / sales AS profit_margin,
        year(order_date) AS order_year,
        month(order_date) AS order_month,
        SUM(sales) OVER (
            PARTITION BY customer_id
        ) AS customer_lifetime_sales
    FROM superstore
);
""")

### Customer Tier Validation

A sample of customers and their corresponding `customer_lifetime_sales` and `customer_tier` values was reviewed to verify that the classification logic was applied correctly. The results confirmed that customers with higher lifetime sales were appropriately assigned to higher value tiers.

In [12]:
con.execute('''
SELECT DISTINCT
        customer_name,
        segment,
        customer_lifetime_sales,
        customer_tier
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,segment,customer_lifetime_sales,customer_tier
0,Sean Miller,Home Office,25043.050,Big Fish
1,Tamara Chand,Corporate,19052.218,Big Fish
2,Raymond Buch,Consumer,15117.339,Big Fish
3,Tom Ashbrook,Home Office,14595.620,Big Fish
4,Adrian Barton,Consumer,14473.571,Big Fish
5,Ken Lonsdale,Consumer,14175.229,Big Fish
6,Sanjit Chand,Consumer,14142.334,Big Fish
7,Hunter Lopez,Consumer,12873.298,Big Fish
8,Sanjit Engle,Consumer,12209.438,Big Fish
9,Christopher Conant,Consumer,12129.072,Big Fish
